# Document Question Answering System — RAG

# Libraries Installation

In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf datasets rank-bm25 accelerate groq python-docx

# 1. Libraries

In [2]:
import os
import re
import time
import textwrap
from getpass import getpass
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from groq import Groq

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", device)

Torch device: cpu


### device: i've used "torch.cuda.is_available()" to check whether the device has GPU as the "BAAI/bge-base-en-v1.5" takes lot of time when it is executed on CPU. So i've shifted the runtime in colab to GPU. But if this file is exectued locally and there is no GPU still this file will exectue without error

# 2. Data Loading (PDF / TXT / DOCX at any path)

In [3]:
from pathlib import Path
from docx import Document

def clean_text(text):
    text = re.sub(r"\s+\n", "\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


directory = Path("")
print("Current Path is: ", "Where this file is place" if directory.name=="" else directory)
files = [f for f in directory.iterdir() if f.is_file() and f.suffix.lower() in {".pdf", ".txt", ".docx"}]

while True:
  print("\n","="*60)
  print("\t Data Loading: ")
  print("="*60)
  print("\nEnter the file Number to load: \n")

  if(len(files)==0):
      print("No PDF or txt file exist at the Path")
  else:
      print("Files in Path:")
      for i,file in enumerate(files):
          print(i+1,file)
  inp = int(input("\nEnter: "))
  if(inp>len(files)):
      print("Invalid input")
  else:
      file_name = files[inp-1].name
      if(file_name.lower().endswith(".pdf")):
          reader = PdfReader(file_name)
          pages = [page.extract_text() or "" for page in reader.pages]
          document_text = clean_text("\n".join(pages))
          break
      elif(file_name.lower().endswith(".txt")):
          with open(file_name, "r", encoding="utf-8") as f:
                document_text = clean_text(f.read())
                break
      elif(file_name.lower().endswith(".docx")):
          doc = Document(file_name)
          paragraphs = [p.text for p in doc.paragraphs]
          document_text = clean_text("\n".join(paragraphs))
          break
      else:
          print("Invalid input/ File Type")

print(f"Loaded document with {len(document_text)} characters.\n")
print(document_text[:500], "...")

Current Path is:  Where this file is place

	 Data Loading: 

Enter the file Number to load: 

Files in Path:
1 RAG_converted.docx
2 RAG_converted.pdf
3 RAG.txt

Enter: 1
Loaded document with 2412 characters.

Renewable Energy: An Overview
Renewable energy comes from natural sources that replenish themselves faster
than they are consumed. Unlike fossil fuels, which take millions of years to
form and release carbon dioxide when burned, renewable sources produce little
to no greenhouse gas emissions during operation. The main types of renewable
energy are solar, wind, hydroelectric, geothermal, and biomass.
Solar Energy
Solar power captures energy from sunlight using photovoltaic (PV) panels or
concentr ...


There are 3 files extentions that can be used to load content
* PDF file that ends with .pdf extension
* Text file that ends with .txt extension
* Word file that end with .docx extension

Once the text is loaded it will be cleaned using "clean_text()" function


# 3. Chunking

In [4]:
TARGET_WORDS = 130
OVERLAP_SENTENCES = 1

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def chunk_text(text, target_words=TARGET_WORDS, overlap_sentences=OVERLAP_SENTENCES):
    sentences = split_sentences(text)
    chunks, current, count = [], [], 0
    for sent in sentences:
        w = len(sent.split())
        if count + w > target_words and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            count = sum(len(s.split()) for s in current)
        current.append(sent)
        count += w
    if current:
        chunks.append(" ".join(current))
    return chunks

def simple_chunk_text(text, chunk_size=120, chunk_overlap=30):
    words = text.split()
    chunks, start, step = [], 0, chunk_size - chunk_overlap
    while start < len(words):
        piece = " ".join(words[start:start + chunk_size]).strip()
        if piece:
            chunks.append(piece)
        start += step
    return chunks

chunks = chunk_text(document_text)
print(f"Document split into {len(chunks)} sentence-aware chunks.\n")
print("Example chunk:\n")
print(textwrap.fill(chunks[0], width=88))

Document split into 4 sentence-aware chunks.

Example chunk:

Renewable Energy: An Overview Renewable energy comes from natural sources that replenish
themselves faster than they are consumed. Unlike fossil fuels, which take millions of
years to form and release carbon dioxide when burned, renewable sources produce little
to no greenhouse gas emissions during operation. The main types of renewable energy are
solar, wind, hydroelectric, geothermal, and biomass. Solar Energy Solar power captures
energy from sunlight using photovoltaic (PV) panels or concentrated solar power systems.
Photovoltaic panels convert light directly into electricity using semiconductor
materials, usually silicon. Solar energy is abundant and increasingly cheap, but it is
intermittent: it only generates power when the sun is shining, which makes energy
storage important. Wind Energy Wind turbines convert the kinetic energy of moving air
into electricity.


Chunking in RAG is the approch that is used to break the large document text into smaller meaningful segments which we say "chunks" before they are embedded. This is the important steps which allows RAG to find information lies in which piece of txt. Its size of matter:
* if the Chunk is too large it will cause irrelevant text in the answer which is not been asked.
* if the size is too small it will cause the single information to be splitted into 2 chunk resulting in half information what the question is asked.

> - The "chunk_text" is **sentence-aware**. It splits the text into whole sentences then packs sentences together until a chunk is about 130 words which gives the embedding model cleaner, more coherent meaning to encode and makes retrieval more accurate.
> - The "simple_chunk_text" is for the experiments section can show the difference that why simple word count chunking is not that accurate comparatively.


# 4. Embedding (stronger model, GPU-accelerated)

In [5]:
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
    if "bge" in EMBED_MODEL_NAME.lower() else ""
)
chunk_embeddings = embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True, batch_size=32).astype("float32")
embedding_dim = chunk_embeddings.shape[1]
print(f"Model: {EMBED_MODEL_NAME}  dimension: {embedding_dim}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Model: BAAI/bge-base-en-v1.5  dimension: 768


An embedding model processes a text chunk and converts it into a numerical vector so that the texts with similar meanings are positioned closer to each other in the vector space. I've use the "BAAI/bge-base-en-v1.5" embedding model because it is lightweight and can run efficiently on a free Google Colab GPU. The model generates 768 dimensional vectors allowing it to capture semantic and contextual relationships within the text easily. This model is capabale of improving the accuracy of downstream tasks such as retrieval and similarity search.


# 5. Vector Database (FAISS)

In [6]:
faiss.normalize_L2(chunk_embeddings)
index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print(f"FAISS vectors size is {index.ntotal}")

FAISS vectors size is 4


### Now these chunk embedding are saved into vectors and it is the fast way to search them when placed in vector for this i've used FAISS. FAISS is built to search thorough large chunks in a blink.

# 6. Query Processing

In [7]:
def embed_query(query):
    q = embed_model.encode([QUERY_INSTRUCTION + query],convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    return q

print("Query vector shape:", embed_query("How does solar power work?").shape)

Query vector shape: (1, 768)


### When we perform retrieval using a query/question that the question has to be embedded the same way as the documents same model, same normalisation. Once the question is a vectored finding relevant chunks becomes a pure "nearest points" problem which FAISS solves in a blink.

# 7. Context Retrieval

In [8]:
def retrieve(query, k=4):
    k = min(k, index.ntotal)
    q = embed_query(query)
    scores, idxs = index.search(q, k)
    results = []
    for rank, (i, s) in enumerate(zip(idxs[0], scores[0]), start=1):
        if i < 0:
            continue
        results.append({
            "rank": rank,
            "score": float(s),
            "chunk": chunks[int(i)],
            "chunk_id": int(i),
        })
    return results

for r in retrieve("How does solar power work?", k=3):
    print(f"\n[Rank {r['rank']}] score={r['score']:.3f}, chunk {r['chunk_id']}\n")
    print(textwrap.fill(r["chunk"], width=170))
    print("="*170)


[Rank 1] score=0.665, chunk 1

Wind Energy Wind turbines convert the kinetic energy of moving air into electricity. Large turbines are often grouped together in wind farms, either onshore or offshore.
Offshore wind is stronger and more consistent, but building and maintaining turbines at sea is more expensive. Wind is also intermittent and depends on weather
conditions. Hydroelectric Power Hydroelectric power generates electricity from flowing or falling water, usually by releasing water from a dam through turbines. It is one
of the oldest and most reliable renewable sources and can be adjusted quickly to meet demand. However, large dams can disrupt ecosystems and displace communities.
Geothermal Energy Geothermal energy taps heat from beneath the Earth's surface. This heat can be used directly for heating buildings or to produce steam that drives
turbines.

[Rank 2] score=0.661, chunk 0

Renewable Energy: An Overview Renewable energy comes from natural sources that replenish themselv

### This is the "R" in RAG is Retrival. It means retriving the information from the chunks. The score on the answers states how confident we are with answers. A high score means the document contains a real answer. A weak score is means that the question might not be answered from the content we have passed.

# 8. Answer Generation (Groq API)

In [9]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
GEN_MODEL = "openai/gpt-oss-20b"

def build_prompt(query, retrieved):
    context = "\n\n".join(f"[{i+1}] {r['chunk']}" for i, r in enumerate(retrieved))
    return (
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

def generate_answer(query, retrieved, max_tokens=400):
    prompt = build_prompt(query, retrieved)
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[{"role": "system",
             "content": "You are a precise assistant that answers strictly from the provided context."},
            {"role": "user", "content": prompt}],
        temperature=0, max_tokens=max_tokens)
    return resp.choices[0].message.content.strip()

demo = retrieve("How does solar power work?", k=4)
print(generate_answer("How does solar power work?", demo))

Enter your Groq API key: ··········
Solar power works by capturing energy from sunlight using photovoltaic (PV) panels or concentrated solar power systems. Photovoltaic panels convert light directly into electricity using semiconductor materials, usually silicon.


### For the LLM I've used the Groq API key from "https://groq.com?utm_source=chatgpt.com". I've used Groq because it is fast and proivdes more tokens at free tier. Earlier i was using Gemini API key but it's tokens were burning too quickly make it less reliable that why i shifted to Groq.


# 9. Pipeline

In [10]:
def ask(question, k=4, show_context=False):
    retrieved = retrieve(question, k=k)
    answer = generate_answer(question, retrieved)
    print("\nQ:", question)
    print("\nA:", textwrap.fill(answer, width=170))
    if show_context:
        print("\nRetrieved context:")
        for r in retrieved:
            print(f" - (score {r['score']:.3f}) {textwrap.fill(r['chunk'][:120], width=170)}...")
    r=retrieved[0]
    return answer, round(r['score'], 3), r['chunk_id']

answer = ask("What is geothermal energy and where can it be used?", show_context=True)


Q: What is geothermal energy and where can it be used?

A: Geothermal energy is the heat that comes from beneath the Earth’s surface. It can be used directly to heat buildings or to produce steam that drives turbines to generate
electricity. Because it requires accessible underground heat, it is most practical in regions such as volcanic areas where the geothermal resource is readily available.

Retrieved context:
 - (score 0.746) This heat can be used directly for heating buildings or to produce steam that drives turbines. Geothermal is highly reli...
 - (score 0.712) Wind Energy Wind turbines convert the kinetic energy of moving air into electricity. Large turbines are often grouped to...
 - (score 0.630) Renewable Energy: An Overview Renewable energy comes from natural sources that replenish themselves faster than they are...
 - (score 0.501) Key challenges include energy storage, upgrading electrical grids, and lowering costs further. Battery technology, smart...


### Pipeline function: It performs ingest, chunking, embeding, storing(vetoring), retrieval, generating in single call. The "show_context=True" parameter lets us see which chunks the model was used to an answer.

# 10. Asking multiple questions (Validation)

In [11]:
test_questions = [
    "What are the main types of renewable energy?",
    "Why is solar energy considered intermittent?",
    "What is the difference between onshore and offshore wind?",
    "What are the downsides of hydroelectric dams?",
    "What is biomass energy made from?",
    "What is the capital of France?",   # deliberately out-of-document
]

print("VALIDATION LOG")
print("=" * 88)
for q in test_questions:
    answer, score, chunk_id = ask(q)
    top_score = score if score else 0.0
    print(f">>> top_retrieval_score={top_score:.3f}, chunks_used=",chunk_id," <<<\n")
    print("=" * 88)

VALIDATION LOG

Q: What are the main types of renewable energy?

A: The main types of renewable energy are solar, wind, hydroelectric, geothermal, and biomass.
>>> top_retrieval_score=0.828, chunks_used= 0  <<<


Q: Why is solar energy considered intermittent?

A: Solar energy is considered intermittent because it only generates power when the sun is shining.
>>> top_retrieval_score=0.576, chunks_used= 1  <<<


Q: What is the difference between onshore and offshore wind?

A: Offshore wind is stronger and more consistent than onshore wind, but building and maintaining turbines at sea is more expensive.
>>> top_retrieval_score=0.634, chunks_used= 1  <<<


Q: What are the downsides of hydroelectric dams?

A: The downsides of hydroelectric dams mentioned in the context are that large dams can disrupt ecosystems and displace communities.
>>> top_retrieval_score=0.563, chunks_used= 1  <<<


Q: What is biomass energy made from?

A: Biomass energy is made from organic materials such as wood, c

> This validation block is our evidence that the RAG is working. I've pushed a batch of questions through and log both the answer and the top retrieval score with chunk_id for each question. And the last question is the to check that is our RAG is correctly working because this question cannot be answered through the content we have passed "What is the capital of France?" it isn't there in renewable-energy document and its low score reflects that. A RAG should not invent answer rather it should explicitly state "I don't know" and that's what RAG is doing that's the sign of good RAG also the score drop too.

# 11. System Metrics Report

In [12]:
chunk_lengths = [len(c.split()) for c in chunks]

print("METRICS REPORT")
print("=" * 60)
print("1. About Document")
print(f"Total characters : {len(document_text)}")
print(f"Total words      : {len(document_text.split())}\n")
print("2. Chunking")
print(f"Method used         : sentence-aware packing")
print(f"Target words/chunk  : {TARGET_WORDS}")
print(f"Sentence overlap    : {OVERLAP_SENTENCES}")
print(f"Number of chunks    : {len(chunks)}")
print(f"Avg words per chunk : {np.mean(chunk_lengths):.1f}")
print(f"Min/Max words       : {min(chunk_lengths)} / {max(chunk_lengths)}\n")
print("3. Embeddings")
print(f"Model Used           : {EMBED_MODEL_NAME}")
print(f"Embedding dimension  : {embedding_dim}")
print(f"Total vectors stored : {index.ntotal}\n")
print("4. Vector")
print(f"Library    : FAISS")
print(f"Index type : IndexFlatIP (cosine via normalised IP)\n")
print("5. Language model (LLM)")
print(f"API         : Groq API")
print(f"Model       : {GEN_MODEL}")
print(f"Temperature : 0")
print("=" * 60)

METRICS REPORT
1. About Document
Total characters : 2412
Total words      : 354

2. Chunking
Method used         : sentence-aware packing
Target words/chunk  : 130
Sentence overlap    : 1
Number of chunks    : 4
Avg words per chunk : 98.8
Min/Max words       : 28 / 127

3. Embeddings
Model Used           : BAAI/bge-base-en-v1.5
Embedding dimension  : 768
Total vectors stored : 4

4. Vector
Library    : FAISS
Index type : IndexFlatIP (cosine via normalised IP)

5. Language model (LLM)
API         : Groq API
Model       : openai/gpt-oss-20b
Temperature : 0


# 12. Experiments — chunking comparison, hybrid search, re-ranking

## Exp 1: Simple vs Sentence-aware chunking

In [13]:
simple = simple_chunk_text(document_text)
smart = chunk_text(document_text)
print("CHUNKING COMPARISON")
print(f"  simple (word-cut)     : {len(simple)} chunks")
print(f"  smart (sentence-aware): {len(smart)} chunks")
print(f"  simple sample ending  : ...{simple[0][-70:]}")
print(f"  smart sample ending  : ...{smart[0][-70:]}   <- ends on a full sentence\n")

CHUNKING COMPARISON
  simple (word-cut)     : 4 chunks
  smart (sentence-aware): 4 chunks
  simple sample ending  : ...torage important. Wind Energy Wind turbines convert the kinetic energy
  smart sample ending  : ...nd turbines convert the kinetic energy of moving air into electricity.   <- ends on a full sentence



### Exp 2: Hybrid search: BM25 + Vector scores

In [14]:
tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def hybrid_retrieve(query, k=4, alpha=0.5):
    q = embed_query(query)
    v_scores, v_idx = index.search(q, index.ntotal)
    vec = np.zeros(len(chunks))
    for i, s in zip(v_idx[0], v_scores[0]):
        if i >= 0:
            vec[i] = s
    kw = np.array(bm25.get_scores(query.lower().split()))
    if kw.max() > kw.min():
        kw = (kw - kw.min()) / (kw.max() - kw.min())
    combined = alpha * vec + (1 - alpha) * kw
    top = np.argsort(combined)[::-1][:k]
    return [{"rank": r + 1, "score": float(combined[i]),"chunk": chunks[i], "chunk_id": int(i)} for r, i in enumerate(top)]

print("HYBRID retrieval for: 'offshore wind'")
for r in hybrid_retrieve("offshore wind", k=3):
    print(f"  [{r['rank']}] {r['score']:.3f} | {textwrap.fill(r['chunk'][:120], width=170)}...")

HYBRID retrieval for: 'offshore wind'
  [1] 0.815 | Wind Energy Wind turbines convert the kinetic energy of moving air into electricity. Large turbines are often grouped to...
  [2] 0.249 | Renewable Energy: An Overview Renewable energy comes from natural sources that replenish themselves faster than they are...
  [3] 0.243 | This heat can be used directly for heating buildings or to produce steam that drives turbines. Geothermal is highly reli...


## Exp 3: Re-ranking using cross-encoder

In [15]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def retrieve_and_rerank(query, k=4, fetch=8):
    candidates = retrieve(query, k=fetch)
    pairs = [(query, c["chunk"]) for c in candidates]
    scores = reranker.predict(pairs)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:k]

print("RE-RANKED retrieval for: 'offshore wind'")
for r in retrieve_and_rerank("offshore wind", k=3):
    print(f"  rerank={r['rerank_score']:.3f} | {textwrap.fill(r['chunk'][:120], width=170)}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RE-RANKED retrieval for: 'offshore wind'
  rerank=4.855 | Wind Energy Wind turbines convert the kinetic energy of moving air into electricity. Large turbines are often grouped to...
  rerank=-7.182 | Renewable Energy: An Overview Renewable energy comes from natural sources that replenish themselves faster than they are...
  rerank=-10.128 | This heat can be used directly for heating buildings or to produce steam that drives turbines. Geothermal is highly reli...


Three experiments each aimed at getting more accuracy.
- **Exp-1**: Chunking smartly trying to preserve the context — printing the tail of a simple chunking method (it often cuts mid-sentence) smart way is using a sentence-aware one (always ending cleanly at fullstop) makes the improvement preserving the context by keeping full sentence.
- **Exp-2**: Hybrid search blending the semantic vector score with a classic BM25 keyword score through an "alpha" dial, so an exact term like "offshore" can't be quietly overlooked by pure meaning-matching.
- **Exp-3**: Re-ranking over-fetches a handful of candidates and then runs a cross-encoder that reads the question and each chunk together for a sharper judgment but this is slower so we only apply it to the top few. And this is getting best result for a "offshore wind" query it stated negative score for 2nd and 3rd chunk and that's accurate because they were related to other topic rather than the question we asked.

## Conclusion

- I've Created a Retrieval-Augmented Generation(RAG) that pull relevant information from .pdf, .txt or .docx file directly. And uses the LLM model to generate answer as per the content passed rather than answering from the knowledge it's being trained upon.

> **The pipeline starts with flexible input method accepting PDFs, plain text, and Word documents. Then i've used a sentence-aware chunking strategy that groups complete sentences up to around 130 words produced more coherent embeddings and performed better than simple word-based chunking. For embeddings, BAAI/bge-base-en-v1.5 provided a good balance performance and speed. Combined with FAISS, it enabled fast and scalable similarity search.**

- Retrieval scores also proved to be a useful confidence signal. In the evaluation, in-scope questions produced accurate, grounded answers, while an out-of-scope question returned a low similarity score and an appropriate "I don't know" response, demonstrating the importance of preventing hallucinations.

- I've used Groq API for the generation stage because it provided reliable and fast response within the free usage tier.

- Lastlu, this project demonstrates how the key components of a modern RAG system document ingestion, chunking, embeddings, vector search, retrieval, and generation work together. These techniques form the foundation of many real world applications including chatbots, knowledge assistants etc.

